In [1]:
import os
import sys
import re
import random
import threading, time
import ast
import numpy as np
import math
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import jinja2
import shutil
import subprocess

from try_parameters import RunData
from run_vpr import *
from process_data import *

from pysat.formula import CNF
import concurrent.futures
import matplotlib.pyplot as plt
import matplotlib

import tempfile
from ConfigSpace import Configuration, ConfigurationSpace

import pandas as pd
from smac import Scenario
from smac import BlackBoxFacade as BBFacade
from smac import HyperparameterOptimizationFacade as HPOFacade
from smac import MultiFidelityFacade as MFFacade
from smac import AlgorithmConfigurationFacade as ACFacade
from smac import RandomFacade as RFacade
from smac import HyperbandFacade as HBFacade

module_path = os.path.abspath(os.path.join(".."))
build_dir = os.path.join(module_path, "uf_results")
blif_data_dir = os.path.join(module_path,"data/blif/")
cnf_data_dir = os.path.join(module_path,"data/cnf/")
act_data_dir = os.path.join(module_path,"data/act/")
arch_data_dir = os.path.join(module_path,"arch/")

VPR_EXECUTABLE = '/home/tinish/vtr-verilog-to-routing-2023/vpr/vpr'
vpr_power_tech_file = '/home/tinish/vtr-verilog-to-routing-2023/vtr_flow/scripts/gf55_vpr_tech_fixed3.xml'

## Specify FPIA Architectural Parameters and VPR Runtime parameters

- NUM_INPUTS: # of input pins of in-memory tiles (that replace conventional FPGA CLBs)
- NUM_OUTPUTS: # of output pins of in-memory tiles (that replace conventional FPGA CLBs)
- FC_IN: fraction of routing tracks each input pin is connected to
- FC_OUT: fraction of routing tracks each output pin is connected to

In [2]:
test_arch_params = {
    "num_inputs": ("NUM_INPUTS", 68),
    "num_outputs": ("NUM_OUTPUTS", 28),
    "pxb_delay": ("PXB_DELAY", 1e-9),
    "fc_in": ("FC_IN", 0.2),
    "fc_out": ("FC_OUT", 0.2),
}
test_run_params = {
    "target_utilization": ("--target_utilization", 1.0),
    "target_pin_utilization": ("--target_ext_pin_util", 1.0),
    "alpha_clustering": ("--alpha_clustering",  0.0),
    "beta_clustering":  ("--beta_clustering",   0.9),
    "place_algorithm": ("--place_algorithm", "bounding_box"),
    "timing_driven_clustering": ("--timing_driven_clustering", "off"),
    "allow_unrelated_clustering": ("--allow_unrelated_clustering", "off"),
    "fail_predictor": ("--routing_failure_predictor", "aggressive"),
    "astar_fac": ("--astar_fac", 2.0),
    #"rr_graph": ("--write_rr_graph", "./rr_graph.xml"),
}

### Run VPR Pack-Place-Route :: FPIA Area + Time Estimates (Routing ONLY) 
 - SET place_mode=3

In [3]:
blif_filename = 'uf50-01.blif'
cnf_file_name = 'uf50-01.cnf'
blif_path = blif_data_dir+blif_filename

formula = CNF(from_file=cnf_data_dir+cnf_file_name)
nvar = formula.nv
nclause = len(formula.clauses)

runner = TilingExperimentManager(blif_path,build_dir+'/vpr_runs_'+blif_filename.replace('.blif',''),arch_data_dir,place_mode=3,vpr_executable=VPR_EXECUTABLE)
runner.nvar = nvar
runner.nclause = nclause

test_r = runner.test_point(test_arch_params, test_run_params, np.random.randint(2**24), -1, archive=True)
print("Status: ",test_r["PR_status"])
print("Routing Channel Width: ",test_r["route_chan_width"])
print("Max Frequency [Routing]: ",test_r["max_frequency"]," MHz")
print("Area [Routing]: ",test_r["routing_area"]) #area is reported in terms of minimum width transistor area (MWTA)

VPR Run Time:  3.0746872425079346
Final Netlist:  /home/tinish/fpia/uf_results/vpr_runs_uf50-01/68_28_-1_(16340407)/uf50-01.net
Found all run data
Status:  Success
Routing Channel Width:  98.0
Max Frequency [Routing]:  544.855  MHz
Area [Routing]:  171426.0


### Run VPR Pack-Place-Route :: FPIA Area + Time Estimates (IMC Tile + Routing)

In [4]:
output_dict = get_analysis(runner,test_arch_params,test_run_params,num_iter=1,route_chan_width=-1,assume_mode="sram")

VPR Run Time:  3.7788596153259277
Final Netlist:  /home/tinish/fpia/uf_results/vpr_runs_uf50-01/68_28_-1_(6802373)/uf50-01.net
Found all run data
Completed Trials:  1  of  1
Successful Routing Trials:  [[1.]]
CLBS USED/TOTAL:  11 / 16
Max Routing Frequency:  450.049  MHz
FA XBAR: ( 70.0 x 25.0 ); BA XBAR: ( 87.0 x 14.0 )
Total Area:  0.06641303148 ; XBAR:  0.03499331148 ; Routing:  0.03141972


### Run VPR Pack-Place-Route :: FPIA Area + Time + Power Estimates (Routing ONLY)
 - SET place_mode=9

In [5]:
blif_filename = 'uf50-01.blif'
cnf_file_name = 'uf50-01.cnf'
act_file_name = 'uf50-01.act'
blif_path = blif_data_dir+blif_filename

formula = CNF(from_file=cnf_data_dir+cnf_file_name)
nvar = formula.nv
nclause = len(formula.clauses)

runner = TilingExperimentManager(blif_path,build_dir+'/vpr_runs_'+blif_filename.replace('.blif',''),arch_data_dir,place_mode=9,vpr_executable=VPR_EXECUTABLE,activity_file=act_data_dir+act_file_name,vpr_power_tech_file=vpr_power_tech_file)
runner.nvar = nvar
runner.nclause = nclause

test_r = runner.test_point_power(test_arch_params, test_run_params, np.random.randint(2**24), -1, archive=True)
print("Status: ",test_r["PR_status"])
print("Routing Channel Width: ",test_r["route_chan_width"])
print("Max Frequency [Routing]: ",test_r["max_frequency"]," MHz")
print("Area [Routing]: ",test_r["routing_area"]) #area is reported in terms of minimum width transistor area (MWTA)
print("Power [Routing]: ",test_r["total_routing_power"]," W")

VPR Time:  3.4236397743225098
/home/tinish/fpia/uf_results/vpr_runs_uf50-01/68_28_-1_(15503772)/uf50-01.net
Found all run data
Found all Power data
Status:  Success
Routing Channel Width:  70.0
Max Frequency [Routing]:  492.043  MHz
Area [Routing]:  158898.0
Power [Routing]:  0.00518085  W


### Run VPR Pack-Place-Route :: FPIA Area + Time + Power Estimates (IMC Tile + Routing)

In [6]:
output_dict = get_power_analysis(runner,test_arch_params,test_run_params,num_iter=1,route_chan_width=-1,assume_mode="sram")

VPR Time:  6.079843997955322
/home/tinish/fpia/uf_results/vpr_runs_uf50-01/68_28_-1_(14806606)/uf50-01.net
Found all run data
Found all Power data
Completed Trials:  1  of  1
Successful Routing Trials:  [[1.]]
CLBS USED/TOTAL:  11 / 16
Max Routing Frequency:  1139.43  MHz
FA XBAR: ( 70.0 x 25.0 ); BA XBAR: ( 87.0 x 14.0 )
Total Area:  0.15321875148 ; XBAR:  0.03499331148 ; Routing:  0.11822544
Total Power:  13.868843499999999 ; XBAR:  13.863674 ; Routing:  0.0051695000000000005


###Automated Architecture Optimization using SMAC

We employ **Sequential Model-Based Algorithm Configuration (SMAC)** to optimize FPGA architecture parameters for improved crossbar utilization.

For each candidate configuration, the framework generates a parameterized FPGA architecture, executes VPR packing, placement, and routing, and evaluates crossbar utilization from the resulting implementation.

The optimization objective is to minimize the logarithmic utilization loss:

$$
L = -\log_{10}(U)
$$

where $U$ denotes crossbar utilization. Failed VPR runs are assigned an infinite loss when no successful evaluation is available.

Depending on the selected optimization mode, SMAC explores the number of crossbar inputs and outputs or the input/output routing connectivity parameters. One VPR evaluation per candidate configuration.

The optimizer returns the incumbent architecture configuration with the best observed objective value.


In [14]:
def do_hyperOPT(runner):
    def loss(config, seed):

        if runner.hyper_opt_type==1:
            num_inputs = int(round(config["num_inputs"]))
            num_outputs = int(round(config["num_outputs"]))
            fc_in = runner.default_fc_in
            fc_out = runner.default_fc_out
            print("Testing at NUM_INPUTS: ",num_inputs)
            print("Testing at NUM_OUTPUTS: ",num_outputs)
        elif runner.hyper_opt_type==2:
            num_inputs = int(runner.default_num_inputs)
            num_outputs = int(runner.default_num_outputs)
            fc_in = config["fc_in"]
            fc_out = config["fc_out"]
            print("Testing at FC_IN: ",fc_in)
            print("Testing at FC_OUT: ",fc_out)
        elif runner.hyper_opt_type==3:
            num_inputs = int(round(config["num_inputs"]))
            num_outputs = int(runner.default_num_outputs)
            fc_in = runner.default_fc_in
            fc_out = runner.default_fc_out
            print("Testing at NUM_INPUTS: ",num_inputs)
        
        test_arch_params = {
                "num_inputs": ("NUM_INPUTS", num_inputs),
                "num_outputs": ("NUM_OUTPUTS", num_outputs),
                "pxb_delay": ("PXB_DELAY", 1e-9),
                "fc_in": ("FC_IN", fc_in),
                "fc_out": ("FC_OUT", fc_out),
            }
        test_run_params = {
            "target_utilization": ("--target_utilization", 1.0),
            "target_pin_utilization": ("--target_ext_pin_util", 1.0),
            "alpha_clustering": ("--alpha_clustering",  0.0),
            "beta_clustering":  ("--beta_clustering",   0.9),
            "place_algorithm": ("--place_algorithm", "bounding_box"),
            "timing_driven_clustering": ("--timing_driven_clustering", "off"),
            "allow_unrelated_clustering": ("--allow_unrelated_clustering", "off"),
            "fail_predictor": ("--routing_failure_predictor", "aggressive"),
            "astar_fac": ("--astar_fac", 2.0),
        }
        
        num_iter = 1
        err_indices = np.ones((num_iter,1))
        loss_arr = np.zeros((num_iter,1))
        dim_v = 0
        dim_c = 0
        for i in range(num_iter):
            test_r = runner.test_point(test_arch_params, test_run_params, np.random.randint(2**24), -1, archive=True)
            if test_r["PR_status"]=="Error":
                err_indices[i] = 0
            else:
                test_r["nvar"] = runner.nvar
                test_r["nclause"] = runner.nclause
                frac_utilization = get_xbar_util_loss(test_r["netlist_filename"],test_r["nvar"])
                nv_int_dim, nv_ext_dim, nc_int_dim, nc_ext_dim, new_xbar_size, _, _, _, _, _ = get_dualXbarSize(test_r["netlist_filename"],test_r["nvar"])
                dim_v = 2*(nv_ext_dim+nv_int_dim)
                dim_c = (nc_ext_dim+nc_int_dim)
                loss_arr[i] = -np.log10(frac_utilization)
                
        if len(np.where(err_indices==1)[0])==0:
            tiling_loss = math.inf
        else:
            tiling_loss = np.mean(loss_arr[np.where(err_indices==1)[0]])
        
        print("log Tiling Loss: ",tiling_loss,"; # CLBs used: ",test_r["clbs_used"],"; XBAR Area: ",test_r["clbs_total"]*dim_v*dim_c*7.47e-6,"; Routing Area: ",test_r["routing_area"]*area_assumption[2]["Afetmin"]*((0.06*0.06)/(1000*1000)),"; (V,C): (",dim_v,",",dim_c,")")
        return tiling_loss

    if runner.hyper_opt_type==1:
        configspace = ConfigurationSpace({
            "num_inputs": (36, 110),
            "num_outputs": (10, 52),
        })
    elif runner.hyper_opt_type==2:
        configspace = ConfigurationSpace({
            "fc_in": (0.1, 0.4),
            "fc_out": (0.1, 0.4),
        })
    elif runner.hyper_opt_type==3:
        configspace = ConfigurationSpace({
            "num_inputs": (30, 80),
        })
        
    scenario = Scenario(configspace, deterministic=False, n_trials=80, walltime_limit=10)
    optimizer = ACFacade(scenario, loss, overwrite=True,logging_level=20)

    incumbent = optimizer.optimize()

    return incumbent

In [18]:
blif_filename = 'uf50-01.blif'
cnf_file_name = 'uf50-01.cnf'
blif_path = blif_data_dir+blif_filename

formula = CNF(from_file=cnf_data_dir+cnf_file_name)
nvar = formula.nv
nclause = len(formula.clauses)

runner = TilingExperimentManager(blif_path,build_dir+'/vpr_runs_'+blif_filename.replace('.blif',''),arch_data_dir,place_mode=3,vpr_executable=VPR_EXECUTABLE)
runner.nvar = nvar
runner.nclause = nclause
runner.default_num_inputs = 68
runner.default_num_outputs = 68
runner.default_fc_in = 0.2
runner.default_fc_out = 0.2
runner.hyper_opt_type = 1

bo_results = do_hyperOPT(runner)
    
print(bo_results)

if runner.hyper_opt_type==1:
    runner.default_num_inputs = bo_results['num_inputs']
    runner.default_num_outputs = bo_results['num_outputs']
elif runner.hyper_opt_type==2:
    runner.default_fc_in = bo_results['fc_in']
    runner.default_fc_out = bo_results['fc_out']
elif runner.hyper_opt_type==3:
    runner.default_num_inputs = bo_results['num_inputs']
        
#df = pd.DataFrame(runners)
#filename = "satlib_hopt_latest"
#df.to_pickle(filename)

[INFO][abstract_initial_design.py:147] Using 1 initial design configurations and 0 additional configurations.
Testing at NUM_INPUTS:  73
Testing at NUM_OUTPUTS:  31
VPR Run Time:  3.774085283279419
Final Netlist:  /home/tinish/fpia/uf_results/vpr_runs_uf50-01/73_31_-1_(8057811)/uf50-01.net
Found all run data
log Tiling Loss:  0.4042740219943416 ; # CLBs used:  10 ; XBAR Area:  0.80484768 ; Routing Area:  0.02924856 ; (V,C): ( 74.0 , 91.0 )
[INFO][abstract_intensifier.py:515] Added config ce98db as new incumbent because there are no incumbents yet.
Testing at NUM_INPUTS:  73
Testing at NUM_OUTPUTS:  31
VPR Run Time:  4.225336790084839
Final Netlist:  /home/tinish/fpia/uf_results/vpr_runs_uf50-01/73_31_-1_(9752236)/uf50-01.net
Found all run data
log Tiling Loss:  0.4042740219943416 ; # CLBs used:  10 ; XBAR Area:  0.80484768 ; Routing Area:  0.03472416 ; (V,C): ( 74.0 , 91.0 )
Testing at NUM_INPUTS:  102
Testing at NUM_OUTPUTS:  13
VPR Run Time:  3.02740478515625
Final Netlist:  /home/ti